## IMPORTATION DES BIBLIOTHEQUES ET PARAMETRES GLOBAUX

In [3]:
#Fonctions de génération de la grille

def degrees_per_km(latitude):
    """Calcule la conversion degrés/km pour une latitude donnée."""
    lat_deg_per_km = 1 / 111.32
    lon_deg_per_km = 1 / (111.32 * math.cos(math.radians(latitude)))
    return lat_deg_per_km, lon_deg_per_km

def create_country_grid_WGS84(gdf, country_code, col_code="shapeGroup", grid_size_km=10, midpoint_lat=None, crop=False, display=False):
    """Génère une grille pour un pays spécifique en WGS84."""
    country = gdf[gdf[col_code] == country_code]
    if country.empty:
        raise ValueError(f"Pays '{country_code}' introuvable dans le GeoDataFrame.")

    def create_grid(country, grid_size, reference_point=(0, 0), midpoint_lat=None, crop=False):
        """Création d'une grille avec une taille définie."""
        ref_x, ref_y = reference_point
        country_geometry = country.geometry.union_all()
        minx, miny, maxx, maxy = country_geometry.bounds
        
        if midpoint_lat is None:
            midpoint_lat = (miny + maxy) / 2
        lat_deg_per_km, lon_deg_per_km = degrees_per_km(midpoint_lat)
        dy, dx = grid_size * lat_deg_per_km, grid_size * lon_deg_per_km
        start_x, start_y = ref_x + ((minx - ref_x) // dx) * dx, ref_y + ((miny - ref_y) // dy) * dy
        
        grid_cells = []
        for x in np.arange(start_x, maxx, dx):
            for y in np.arange(start_y, maxy, dy):
                cell = box(x, y, x + dx, y + dy)
                grid_cells.append({"geometry": cell, "min_lon": x, "min_lat": y, "max_lon": x + dx, "max_lat": y + dy})
        
        grid_gdf = gpd.GeoDataFrame(grid_cells, crs=country.crs)
        
        # === Ajout des rangs ===
        # On se base sur les centres des cellules pour gérer les petits décalages
        grid_gdf["x_center"] = ((grid_gdf["min_lon"] + grid_gdf["max_lon"]) / 2 / dx).round().astype(int)
        grid_gdf["y_center"] = ((grid_gdf["min_lat"] + grid_gdf["max_lat"]) / 2 / dy).round().astype(int)
    
        x_unique = np.sort(grid_gdf["x_center"].unique())
        y_unique = np.sort(grid_gdf["y_center"].unique())
    
        # Mapping pour les rangs
        grid_gdf["rank_x"] = grid_gdf["x_center"].map({v: i for i, v in enumerate(x_unique)})
        grid_gdf["rank_y"] = grid_gdf["y_center"].map({v: i for i, v in enumerate(y_unique)})
    
        # Clé de maille unique
        grid_gdf["cleMaille"] = grid_gdf["rank_x"].astype(str) + "_" + grid_gdf["rank_y"].astype(str)
    
        # Nettoyage colonnes temporaires
        grid_gdf = grid_gdf.drop(columns=["x_center", "y_center"])

        return gpd.clip(grid_gdf, country_geometry) if crop else grid_gdf
                
    grid = create_grid(country, grid_size_km, midpoint_lat=midpoint_lat, crop=crop)
    grid = grid[grid.intersects(country.geometry.union_all())]
    #grid["cell_name"] = grid.apply(lambda cell: f"{grid_size_km}kmE{int(abs(cell.geometry.centroid.x) * 100):05d}N{int(abs(cell.geometry.centroid.y) * 100):05d}{country_code}", axis=1)
    grid["cell_name"] = grid.apply(lambda cell: f"{grid_size_km}km_{cell['cleMaille']}_{country_code}",axis=1)
    grid["country_code"] = country_code

    if display:
        fig, ax = plt.subplots(figsize=(10, 10))
        gdf.plot(ax=ax, edgecolor="black", linewidth=0.5)
        country.plot(ax=ax, edgecolor="red", linewidth=2, facecolor="none")
        grid.plot(ax=ax, color="lightblue", edgecolor="grey", alpha=0.6)
        plt.show()

    return grid

def create_and_rename_grid(source_gdf, country_code, column_name, grid_size_km, cle_geo, crop,display):
    grid = create_country_grid_WGS84(source_gdf, country_code, column_name, grid_size_km, midpoint_lat=None, crop=crop, display=display)
    return grid.rename(columns={'cell_name': cle_geo})

def check_duplicates(grid, grid_name, cle_geo):
    if not grid.empty:
        duplicate_count = grid[cle_geo].duplicated().sum()
        if duplicate_count > 0:
            print(f"⚠️ Attention : {duplicate_count} doublon(s) trouvé(s) dans {grid_name} !")
        else:
            print(f"✅ Aucun doublon trouvé dans {grid_name}.")

def make_geo_keys_unique(grid, cle_geo):
    if not grid.empty:
        grid["cle_geo_unique"] = grid.groupby(cle_geo).cumcount().astype(str)
        grid.loc[grid["cle_geo_unique"] != "0", cle_geo] += "_" + grid["cle_geo_unique"]
        grid.drop(columns=["cle_geo_unique"], inplace=True)  # Supprime la colonne temporaire

def generer_grille_pays(path_data,country_name,grid_size_km,world_terrestre,world_maritime,critere_terrestre="shapeGroup",critere_maritime="ISO_TER1",name_attribute="shapeName",code_attribute="shapeGroup",source="GBIF"):
    cle_geo = f"codeMaille{grid_size_km}Km"
    country_code=world_terrestre[world_terrestre[name_attribute] == country_name][code_attribute].iloc[0]
    # 1. Sélection du pays terrestre et maritime
    country_terrestre = world_terrestre[world_terrestre[critere_terrestre] == country_code]
    #if country_name == "France":
     #   country_terrestre = world_terrestre[world_terrestre["iso_3166_1_alpha_2_codes"] == "FR"]
    country_maritime = world_maritime[world_maritime[critere_maritime] == country_code]
    
    # Fusion des géométries terrestres et maritimes
    geom_terrestre = country_terrestre.geometry.union_all()
    geom_maritime = country_maritime.geometry.union_all() if not country_maritime.empty else None
    
    if geom_maritime is not None:
        geom_fusionnee = geom_terrestre.union(geom_maritime)
    else:
        geom_fusionnee = geom_terrestre
    
    # Calcul du centre de latitude pour ajuster la grille
    minx, miny, maxx, maxy = geom_terrestre.bounds
    midpoint_lat = (miny + maxy) / 2
    
    # 3. Création des grilles terrestre, maritime et combinée
    country_grid_terrestre = create_and_rename_grid(country_terrestre, country_code, critere_terrestre, grid_size_km, cle_geo, crop=True,display=False)
    country_grid_maritime = gpd.GeoDataFrame()
    
    if not country_maritime.empty:
        country_grid_maritime = create_and_rename_grid(country_maritime, country_code, critere_maritime, grid_size_km, cle_geo, crop=True,display=False)
    
    # Création d'un GeoDataFrame pour la géométrie fusionnée
    gdf_fusionne = gpd.GeoDataFrame(geometry=[geom_fusionnee], crs=world_terrestre.crs)
    gdf_fusionne["Code"] = country_code
    
    # Création de la grille combinée
    country_grid_combined = create_and_rename_grid(gdf_fusionne, country_code, "Code", grid_size_km, cle_geo, crop=False,display=False)
    
    #Rendre les nom de mailles uniques
    make_geo_keys_unique(country_grid_terrestre, cle_geo)
    make_geo_keys_unique(country_grid_maritime, cle_geo)
    make_geo_keys_unique(country_grid_combined, cle_geo)
    
    #Vérification s'il y a des doublons
    check_duplicates(country_grid_terrestre, "country_grid_terrestre", cle_geo)
    check_duplicates(country_grid_maritime, "country_grid_maritime", cle_geo)
    check_duplicates(country_grid_combined, "country_grid_combined", cle_geo)
    
    # 5. Sauvegarde des fichiers
    country_grid_terrestre.to_file(generate_grid_path(path_data,country_name, grid_size_km, 'terrestre', cle_geo,source=source), driver="GeoJSON")
    
    if not country_grid_maritime.empty:
        country_grid_maritime.to_file(generate_grid_path(path_data,country_name, grid_size_km, 'maritime', cle_geo,source=source), driver="GeoJSON")
    
    country_grid_combined.to_file(generate_grid_path(path_data,country_name, grid_size_km, 'combined', cle_geo,source=source), driver="GeoJSON")


def creer_grille_pays(path_data,country_name, grid_size_km, world_terrestre, world_maritime, cle_geo,name_attribute="shapeName",code_attribute="shapeGroup",source="GBIF"):
    """Crée la grille pour un pays spécifique et charge les fichiers associés."""
    country_code = world_terrestre[world_terrestre[name_attribute] == country_name][code_attribute].iloc[0]
    generer_grille_pays(path_data,country_name, grid_size_km, world_terrestre, world_maritime, critere_terrestre=code_attribute, critere_maritime="ISO_TER1",source=source)

    path_grid_terrestre = generate_grid_path(path_data,country_name, grid_size_km, 'terrestre', cle_geo,source=source)
    path_grid_maritime = generate_grid_path(path_data,country_name, grid_size_km, 'maritime', cle_geo,source=source)
    path_grid_combined = generate_grid_path(path_data,country_name, grid_size_km, 'combined', cle_geo,source=source)

    country_grid_terrestre = load_grid_file(path_grid_terrestre, "Grid Terrestre")
    country_grid_maritime = load_grid_file(path_grid_maritime, "Grid Maritime")
    country_grid_combined = load_grid_file(path_grid_combined, "Grid Combined")

    return country_grid_terrestre, country_grid_maritime, country_grid_combined


In [4]:
def process_biodiv_data(df,annee_min=1):
    """Nettoie et formate les données de biodiversité."""
    
    # Copier les données pour éviter les modifications sur l'original
    df_cleaned = df.copy()
    
    # Identifier le nombre initial d'espèces et d'observations
    n_especes_entrée = len(df_cleaned[cle_ID].unique())
    n_obs_entrée = len(df_cleaned)
    
    # 🔹 Convertir 'eventDate' en datetime et compléter 'year'
    df_cleaned['eventDate'] = pd.to_datetime(df_cleaned['eventDate'], errors='coerce', utc=True)
    df_cleaned['year'] = df_cleaned['year'].fillna(df_cleaned['eventDate'].dt.year)
    
    # 🔹 Assurez-vous que les coordonnées sont numériques et filtrer les NaN
    df_cleaned['decimalLongitude'] = pd.to_numeric(df_cleaned['decimalLongitude'], errors='coerce')
    df_cleaned['decimalLatitude'] = pd.to_numeric(df_cleaned['decimalLatitude'], errors='coerce')
    df_cleaned = df_cleaned.dropna(subset=['decimalLongitude', 'decimalLatitude', cle_ID]).reset_index(drop=True)

    print(f"➡️  En entrée : {n_especes_entrée} espèces, {n_obs_entrée} observations")

    # Suppression des lignes avec valeurs manquantes pour les colonnes cruciales
    df_cleaned = df_cleaned.dropna(subset=[cle_ID]).reset_index(drop=True)

    # Filtrer les observations où 'occurrenceStatus' est 'PRESENT'
    df_cleaned = df_cleaned[df_cleaned['occurrenceStatus'] == 'PRESENT'].reset_index(drop=True)

    # Convertir l'ID des espèces en entier
    df_cleaned[cle_ID] = df_cleaned[cle_ID].astype(int)
    
    df_cleaned['year'] = pd.to_numeric(df_cleaned['year'], errors='coerce').fillna(0).astype(int)
    if annee_min is not None:
        df_cleaned = df_cleaned[df_cleaned['year'] >= annee_min]

    # Renommer la colonne contenant la maille géographique
    df_cleaned.rename(columns={'grid_name': cle_geo}, inplace=True)

    # Convertir 'individualCount' en numérique et remplacer NaN par 1
    df_cleaned['individualCount'] = pd.to_numeric(df_cleaned['individualCount'], errors='coerce').fillna(1)

    # Calcul des pertes en pourcentage
    perte_especes = 100 - round(len(df_cleaned[cle_ID].unique()) / n_especes_entrée * 100)
    perte_obs = 100 - round(len(df_cleaned) / n_obs_entrée * 100)

    print(f"✅ En sortie : {len(df_cleaned[cle_ID].unique())} espèces (-{perte_especes}%)")
    print(f"✅ En sortie : {len(df_cleaned)} observations (-{perte_obs}%)")

    return df_cleaned

def add_grid_to_country(df_country, grid,cle_geo):
    df_new=df_country.copy()
    """
    Optimized version of adding the corresponding grid cell to each row in df_country based on latitude and longitude.
    
    Parameters:
    - df_country: DataFrame containing the columns 'decimalLatitude' and 'decimalLongitude'.
    - grid: DataFrame containing the grid cells with 'name', 'min_lon', 'min_lat', 'max_lon', and 'max_lat' columns.
    
    Returns:
    - df_country: Updated DataFrame with an additional 'grid_name' column indicating the grid cell for each point.
    """
    """
    # Convert grid bounds to NumPy arrays for efficient vectorized comparison
    min_lons = grid['min_lon'].values
    max_lons = grid['max_lon'].values
    min_lats = grid['min_lat'].values
    max_lats = grid['max_lat'].values
    """
    
    # Assure-toi que grid['geometry'] contient des Polygons
    bounds = grid['geometry'].bounds  # Cela renvoie un DataFrame avec minx, miny, maxx, maxy
    
    min_lons = bounds['minx'].values
    max_lons = bounds['maxx'].values
    min_lats = bounds['miny'].values
    max_lats = bounds['maxy'].values

    
    grid_names = grid[cle_geo].values


    # Initialize an array to store the grid names
    grid_names_for_points = []
    s=0
    n=0
    print(f"➡️ Association des mailles aux données")
    # Iterate over each point in df_country and apply vectorized comparison
    for lon, lat in zip(df_country['decimalLongitude'], df_country['decimalLatitude']):
        # Find the grid cell by comparing the point coordinates with grid bounds
        matching_grid = np.where((min_lons <= lon) & (lon <= max_lons) & (min_lats <= lat) & (lat <= max_lats))[0]
        
        if matching_grid.size > 0:
            grid_names_for_points.append(grid_names[matching_grid[0]])  # Take the first matching grid cell
            if matching_grid.size > 1:
                s=s+1
      
        else:
            grid_names_for_points.append(None)  # No matching grid
            n=n+1
       
    print(f'{s} with several matching grids')
    print(f'{n} with no matching grid')
    # Add the grid names to the DataFrame
    df_new[cle_geo] = grid_names_for_points
    
    return df_new
    
def formater_maille_espece_GBIF(df,cle_geo='codeMaille10Km',cle_ID='cdRef',bornes_temporelles=None):
    df_dico=generer_dictionnaire_taxonomie(df,cle_ID)
    # Convertir la colonne 'year' en int
    df['year'] = pd.to_numeric(df['year'], errors='coerce').fillna(0).astype(int)

    # Choisir des bornes temporelles et assigner une période aux données
    if bornes_temporelles is not None:
        df.loc[:, 'periode'] = pd.cut(df['year'], bins=bornes_temporelles, 
                       labels=[f'Période {i+1}: {bornes_temporelles[i]+1} à {bornes_temporelles[i+1]}' for i in range(len(bornes_temporelles) - 1)],
                       include_lowest=False)  # include_lowest=True inclut la borne inférieureure
        # Compter le nombre de données dans chaque intervalle
        compte_par_periode = df['periode'].value_counts()

         # Compter les occurrences d'observation de chaque taxon pour chaque code et période
        df_maille_espece = df.groupby([cle_geo, cle_ID,'periode'], observed=True).size().reset_index(name='nombreObs')
        #df_maille_espece = df.groupby([cle_geo, cle_ID,'periode'], observed=True)['individualCount'].sum().reset_index(name='nombreObs')
        #eventuellement remplacer ['individualCount'].sum() par .size() 
       
    else:
        # Compter les occurrences d'observation de chaque taxon pour chaque code
        df_maille_espece = df.groupby([cle_geo, cle_ID], observed=True).size().reset_index(name='nombreObs')
        #eventuellement remplacer ['individualCount'].sum() par .size() 
        
    df_maille_espece=pd.merge(df_maille_espece,df_dico,on=cle_ID)
    
    return df_maille_espece
    
# 📌 Fonction pour charger et traiter chaque chunk
def process_chunk(df_biodiv, country_grid_terrestre, country_grid_maritime, country_grid_combined, bornes_temporelles, chunk_number):
    """Prétraitement et enregistrement des données par chunk"""
    
    print(f"\n🔍 Traitement du chunk {chunk_number}...")

    df_biodiv = process_biodiv_data(df_biodiv)

    # 🔹 Ajouter la maille pour chaque catégorie
    if country_grid_terrestre is not None:
        df_terrestre = add_grid_to_country(df_biodiv, country_grid_terrestre, cle_geo).dropna(subset=[cle_geo]).reset_index(drop=True)
        
    if country_grid_maritime is not None:
        df_maritime = add_grid_to_country(df_biodiv, country_grid_maritime, cle_geo).dropna(subset=[cle_geo]).reset_index(drop=True)

    if country_grid_combined is not None:
        df_combined = add_grid_to_country(df_biodiv, country_grid_combined, cle_geo).dropna(subset=[cle_geo]).reset_index(drop=True)

    # 🔹 Regrouper par maille et période
    chaine_bornes = "_".join(map(str, bornes_temporelles))
    
    if country_grid_terrestre is not None:
        df_maille_espece_terrestre = formater_maille_espece_GBIF(df_terrestre, cle_geo, cle_ID, bornes_temporelles)
    if country_grid_maritime is not None:
        df_maille_espece_maritime = formater_maille_espece_GBIF(df_maritime, cle_geo, cle_ID, bornes_temporelles)
    if country_grid_combined is not None:
        df_maille_espece_combined = formater_maille_espece_GBIF(df_combined, cle_geo, cle_ID, bornes_temporelles)

    # 🔹 Ajouter les noms vernaculaires
    dico_noms_vernaculaires = pd.read_csv(path+ r"\TAXO_GBIF\dico_noms_vernaculaires_merged.csv")

    if country_grid_terrestre is not None:
        df_maille_espece_terrestre = pd.merge(df_maille_espece_terrestre, dico_noms_vernaculaires, on=cle_ID, how="left")
    if country_grid_maritime is not None:
        df_maille_espece_maritime = pd.merge(df_maille_espece_maritime, dico_noms_vernaculaires, on=cle_ID, how="left")
    if country_grid_combined is not None:
        df_maille_espece_combined = pd.merge(df_maille_espece_combined, dico_noms_vernaculaires, on=cle_ID, how="left")
    
    # 🔹 Sauvegarde des fichiers
    
    def save_chunk(df, ecosysteme,chunk_number,source="GBIF"):
        
        # Créer le répertoire s'il n'existe pas
        path_save=os.path.join(path_data,source, 'processed',f"{source}_{country_name}")
        os.makedirs(path_save, exist_ok=True)
        df.to_csv(os.path.join(path_save, f"data_{source}_{country_name}_{ecosysteme}_{cle_geo}_{cle_ID}_periodes{chaine_bornes}_{chunk_number}.csv"), index=False)

    if country_grid_terrestre is not None:
        save_chunk(df_maille_espece_terrestre, "terrestre",chunk_number,source=source)
    if country_grid_maritime is not None:
        save_chunk(df_maille_espece_maritime, "maritime",chunk_number,source=source)
    if country_grid_combined is not None:
        save_chunk(df_maille_espece_combined, "combined",chunk_number,source=source)

    # 🔹 Nettoyage mémoire
# 🔹 Nettoyage mémoire
if 'df_biodiv' in locals():
    del df_biodiv
if 'df_terrestre' in locals():
    del df_terrestre
if 'df_maritime' in locals():
    del df_maritime
if 'df_combined' in locals():
    del df_combined
if 'df_maille_espece_terrestre' in locals():
    del df_maille_espece_terrestre
if 'df_maille_espece_maritime' in locals():
    del df_maille_espece_maritime
if 'df_maille_espece_combined' in locals():
    del df_maille_espece_combined

    print(f"\n✅ Chunk {chunk_number} traité et sauvegardé avec succès ! 🎉\n")


def detect_sep(path_fichier, seps=[",",";","\t","|"," "], nrows=5):
    """Détecte automatiquement le séparateur le plus probable."""
    best_sep = None
    best_cols = 0
    
    for sep in seps:
        try:
            df = pd.read_csv(path_fichier, sep=sep, nrows=nrows, engine="python")
            ncols = len(df.columns)
            if ncols > best_cols:
                best_cols = ncols
                best_sep = sep
        except Exception:
            continue
    
    if best_sep is None:
        raise ValueError("Impossible de déterminer un séparateur correct")
    
    print(f"✅ Séparateur choisi : '{best_sep}' avec {best_cols} colonnes")
    return best_sep


def traiter_chunks(path_fichier, colonnes_a_importer,
                   country_grid_terrestre, country_grid_maritime,
                   country_grid_combined, bornes_temporelles):
    """Lit les données par chunks et les traite pour chaque écosystème."""

    
    # 🔍 Détection automatique du séparateur
    sep = detect_sep(path_fichier)
    
    chunk_number = 0
    for df_biodiv in pd.read_csv(
        path_fichier,
        sep=sep,
        quoting=csv.QUOTE_NONE,
        chunksize=10_000_000,
        on_bad_lines='skip',
        usecols=lambda c: c in colonnes_a_importer  # garde seulement les colonnes dispo
    ):
        chunk_number += 1
        process_chunk(df_biodiv, country_grid_terrestre,
                      country_grid_maritime,
                      country_grid_combined,
                      bornes_temporelles,
                      chunk_number)
    return chunk_number


def fusionner_fichiers_par_ecosysteme(country_name, chunk_number, ecosysteme, cle_geo, cle_ID, bornes_temporelles, path_data,source="GBIF"):
    """Fusionne les fichiers pour un écosystème donné et génère le fichier final."""
    chaine_bornes = "_".join(map(str, bornes_temporelles))
    df_final = pd.DataFrame()
    fichiers_trouves = False 

    for i in range(1, chunk_number + 1):
        file_path = os.path.join(path_data,source, 'processed',f"{source}_{country_name}", f"data_{source}_{country_name}_{ecosysteme}_{cle_geo}_{cle_ID}_periodes{chaine_bornes}_{i}.csv")
        if os.path.exists(file_path):
            fichiers_trouves = True
            df_temp = pd.read_csv(file_path)
            df_final = pd.concat([df_final, df_temp], ignore_index=True)
    # Si aucun fichier n'a été trouvé, on arrête la fonction
    if not fichiers_trouves:
        print(f"⚠️ Aucun fichier trouvé pour {country_name} - {ecosysteme} - {cle_geo}. Aucune fusion effectuée.")
        return df_final

    # Regroupement des observations
    df_maille_espece = df_final.groupby([cle_geo, cle_ID, 'periode'], observed=True)['nombreObs'].sum().reset_index()

    # Génération du dictionnaire taxonomique
    df_dico = generer_dictionnaire_taxonomie(df_final, cle_ID)

    # Fusion des données
    df_maille_espece = pd.merge(df_maille_espece, df_dico, on=cle_ID)

    # Sauvegarde du fichier fusionné
    final_file_path = os.path.join(path_data,source, 'processed',f"{source}_{country_name}", f"data_{source}_{country_name}_{ecosysteme}_{cle_geo}_{cle_ID}_periodes{chaine_bornes}.csv")
    df_maille_espece.to_csv(final_file_path, index=False)

    # Suppression des fichiers chunk après fusion
    for i in range(1, chunk_number + 1):
        file_path = os.path.join(path_data,source,'processed',f"{source}_{country_name}",  f"data_{source}_{country_name}_{ecosysteme}_{cle_geo}_{cle_ID}_periodes{chaine_bornes}_{i}.csv")
        if os.path.exists(file_path):
            os.remove(file_path)

    return df_maille_espece




In [5]:
def calcul_largeur_maille_pays(country_name, world_terrestre, path_fichier, n_donnees_par_maille):
    """
    Calcule la largeur de maille recommandée pour un pays donné à partir d'un fichier CSV.
    
    Arguments :
    - country_name : nom du pays (str)
    - world_terrestre : GeoDataFrame contenant les géométries des pays
    - path_fichier : chemin vers le fichier CSV
    - n_donnees_par_maille : nombre de données souhaité par maille
    
    Retour :
    - largeur_maille_recommande : largeur de maille en km
    """
    
    # Sélection du pays
    country_shape = world_terrestre[world_terrestre['shapeName'] == country_name]
    if country_shape.empty:
        raise ValueError(f"❌ Pays {country_name} introuvable dans world_terrestre")
    
    # Reprojection en projection métrique pour calculer la surface en km²
    country_shape_metric = country_shape.to_crs(epsg=3857)  # projection métrique
    surface_km2 = country_shape_metric.geometry.area.sum() / 10**6

    # Comptage des lignes du fichier CSV
    with open(path_fichier, "r", encoding="utf-8") as f:
        nb_lignes_sum = sum(1 for _ in f)

    # Calcul de la densité de données
    densite_donnees = nb_lignes_sum / surface_km2
    
    # Calcul de la largeur de maille recommandée
    largeur_maille_recommande = round(math.sqrt(n_donnees_par_maille / densite_donnees))
    
    print(f"La largeur de maille recommandée pour {country_name} est de {largeur_maille_recommande:.2f} km")
    return largeur_maille_recommande

def rename_sig_files(country_name, grid_size_km, path_data, source):
    """
    Duplique et renomme les fichiers SIG :
    grid_{country}_{ecosysteme}_codeMaille{grid_size}Km.geojson
    en
    grid_{country}_{ecosysteme}_codeMaille{grid_size}Km_rec.geojson
    sans utiliser glob ni shutil.
    """
    # Chemin du dossier SIG
    sig_folder = os.path.join(path_data, source, "processed",f"{source}_{country_name}")
    
    if not os.path.exists(sig_folder):
        print(f"⚠️ Le dossier SIG n'existe pas : {sig_folder}")
        return
    
    # Parcours de tous les fichiers du dossier
    for file_name in os.listdir(sig_folder):
        suffix = f"codeMaille{grid_size_km}Km.geojson"
        if file_name.endswith(suffix):
            old_path = os.path.join(sig_folder, file_name)
            new_file_name = file_name.replace(suffix, f"codeMaille{grid_size_km}Km_rec.geojson")
            new_path = os.path.join(sig_folder, new_file_name)
            
            # Dupliquer le fichier en lisant et écrivant
            with open(old_path, 'rb') as f_src:
                content = f_src.read()
            with open(new_path, 'wb') as f_dst:
                f_dst.write(content)
            
            print(f"✅ Fichier dupliqué : {file_name} → {new_file_name}")



In [6]:
def traiter_pays_et_maille(
    country_name, grid_size_km, bornes_temporelles, path, cle_geo, cle_ID,
    source="GBIF", fusion=False,
    var_obs='nombreObs', seuil_fusion=1000, methode_fusion="barycentre", max_distance=100
):
    """Exécute le traitement complet pour un pays et une taille de maille donnés."""
    
    world_terrestre, world_maritime = charger_donnees_geo(path)
    
    country_grid_terrestre, country_grid_maritime, country_grid_combined = creer_grille_pays(
        path_data, country_name, grid_size_km, world_terrestre, world_maritime, cle_geo, source=source
    )

    path_fichier = os.path.join(path_data, source,"raw", f"{source}_{country_name}", f"extract{source}_{country_name}.csv")
    
    colonnes_a_importer = [
        'kingdom', 'phylum', 'class', 'order', 'family', 'genus', 'species',
        'verbatimScientificName', 'taxonRank', 'countryCode', 'occurrenceStatus',
        'individualCount', 'decimalLongitude', 'decimalLatitude', 'eventDate',
        'speciesKey', 'occurrenceID', 'year'
    ]
    
    chunk_number = traiter_chunks(
        path_fichier, colonnes_a_importer,
        country_grid_terrestre, country_grid_maritime, country_grid_combined,
        bornes_temporelles
    )

    processed_path = os.path.join(path_data, "processed")
    os.makedirs(processed_path, exist_ok=True)

    for ecosysteme in ['maritime', 'combined', 'terrestre']:
        df_final = fusionner_fichiers_par_ecosysteme(
            country_name, chunk_number, ecosysteme,
            cle_geo, cle_ID, bornes_temporelles, path_data, source=source
        )
    
        # --- Fusion si demandée ---
        if fusion and ecosysteme == 'terrestre':

            # --- choisir la grille selon l'écosystème ---
            if ecosysteme == 'maritime':
                grid = country_grid_maritime
            elif ecosysteme == 'combined':
                grid = country_grid_combined
            elif ecosysteme == 'terrestre':
                grid = country_grid_terrestre
            else:
                raise ValueError(f"Ecosystème inconnu : {ecosysteme}")
            df_obs = df_final.groupby(cle_geo, as_index=False)[var_obs].sum()
            
            # merge du nombre d'observations
            merged = grid.merge(
                df_obs,   # somme des nombreObs par maille
                on=cle_geo,
                how="left"
            )

            # --- Affichage initial ---
            merged[var_obs] = merged[var_obs].fillna(0)      # remplacer les NaN par 0
            merged[var_obs] = merged[var_obs].astype(float) # s'assurer que c'est float
            merged['coords'] = merged.apply(lambda row: [(row['rank_x'], row['rank_y'])], axis=1)

            
            print(f"⚡ Application de la fusion des cellules ({methode_fusion})...")
            merged_fusion = fusion_cells_by_obs(
                merged,
                cle_geo, var_obs,
                seuil=seuil_fusion,
                methode=methode_fusion,
                max_distance=max_distance
            )

            # Colonnes à garder
            colonnes_a_garder = [
                'min_lon', 'min_lat', 'max_lon', 'max_lat', 
                'rank_x', 'rank_y', 'cleMaille', cle_geo, 
                'country_code', 'geometry'
            ]
            
            # créer le dataframe grid_fusion
            grid_fusion = merged_fusion[colonnes_a_garder].copy()
            
            # définir le chemin pour sauvegarde
            grid_fusion_path = generate_grid_path(
                path_data, country_name, grid_size_km, ecosysteme, cle_geo, source=source, fusion=True,seuil_fusion=seuil_fusion
            )
            
            # sauvegarde au format GeoJSON
            grid_fusion.to_file(grid_fusion_path, driver="GeoJSON")
            
            print(f"✅ grid_fusion sauvegardé dans : {grid_fusion_path}")
            
            df_maille_espece=apply_fusion_to_biodiv(df_final, merged_fusion, cle_geo=cle_geo, cle_ID=cle_ID, var_obs=var_obs)
            
             # Sauvegarde du fichier fusionné
            final_file_path = os.path.join(path_data,source,'processed',f"{source}_{country_name}", f"data_{source}_{country_name}_{ecosysteme}_{cle_geo}Adapt{seuil_fusion}_{cle_ID}_periodes{chaine_bornes}.csv")
            df_maille_espece.to_csv(final_file_path, index=False)

                                   
    
    print(f"✅ Traitement complet pour {country_name} avec une taille de maille {grid_size_km} km")
    return df_final



### Fusion

In [7]:
def fusion_cells_by_obs(gdf, cle_geo, var_obs, seuil=1000, methode="barycentre", max_distance=100,display=True):
    """
    Fusionne les cellules tant que nombreObs < seuil et garde la trace des cellules fusionnées.
    Si le total est < seuil, tout finit par être fusionné en une seule cellule avec la vraie somme.
    """
    gdf = gdf.copy()
    n_fusion=0
    
    # colonne pour garder la trace des cellules fusionnées
    gdf['fusion'] = gdf[cle_geo].apply(lambda x: [x])
    
    if methode == "barycentre":
        # calculer les barycentres
        gdf['centroid'] = gdf['geometry'].centroid
    
    # boucle principale
    while len(gdf) > 1 and gdf[var_obs].min() < seuil:
        n_fusion=n_fusion+1
        print(f"Fusion n°{n_fusion}, n_min= {gdf[var_obs].min()}")
        idx_min = gdf[var_obs].idxmin()
        cell = gdf.loc[idx_min]

        if methode == "barycentre":
            centroid_cell = cell['centroid']
            others = gdf.drop(idx_min)
            if others.empty:
                break  # plus rien à fusionner
            others['distance'] = others['centroid'].apply(lambda c: centroid_cell.distance(c))
            neighbor_idx = others['distance'].idxmin()
        else:
            x_min, y_min = cell['rank_x'], cell['rank_y']
            neighbors = pd.DataFrame()
            for dist in range(1, max_distance+1):
                neighbors = gdf[
                    ((abs(gdf['rank_x'] - x_min) == dist) & (gdf['rank_y'] == y_min)) |
                    ((abs(gdf['rank_y'] - y_min) == dist) & (gdf['rank_x'] == x_min))
                ].drop(idx_min, errors='ignore')
                neighbors = neighbors[neighbors[var_obs].notna()]
                if not neighbors.empty:
                    break
            if neighbors.empty:
                break  # rien à fusionner

            if methode == "max":
                val = neighbors[var_obs].max()
                candidates = neighbors[neighbors[var_obs] == val]
            elif methode == "min":
                val = neighbors[var_obs].min()
                candidates = neighbors[neighbors[var_obs] == val]
            elif methode == "random":
                candidates = neighbors
            else:
                raise ValueError("methode doit être 'max', 'min', 'random' ou 'barycentre'")
            
            neighbor_idx = np.random.choice(candidates.index)

        # fusion
        gdf.at[neighbor_idx, var_obs] += gdf.at[idx_min, var_obs]
        gdf.at[neighbor_idx, 'geometry'] = gdf.at[neighbor_idx, 'geometry'].union(gdf.at[idx_min, 'geometry'])
        gdf.at[neighbor_idx, 'fusion'] = gdf.at[neighbor_idx, 'fusion'] + gdf.at[idx_min, 'fusion']

        # supprimer cellule fusionnée
        gdf = gdf.drop(idx_min)
        
        # mettre à jour centroid si nécessaire
        if methode == "barycentre":
            gdf.loc[neighbor_idx, 'centroid'] = gdf.loc[neighbor_idx, 'geometry'].centroid
    
    # supprimer colonne temporaire centroid
    if 'centroid' in gdf.columns:
        gdf = gdf.drop(columns=['centroid'])
    
    return gdf.reset_index(drop=True)

def apply_fusion_to_biodiv(df_biodiv, merged_fusion, cle_geo='codeMaille10Km', cle_ID='speciesKey', var_obs='nombreObs'):
    """
    Applique les fusions de cellules de merged_fusion à df_biodiv,
    en remplaçant les codes des cellules originales par le code de la cellule dominante,
    puis en regroupant par cellule, espèce et période en sommant les observations.
    
    Paramètres :
    - df_biodiv : DataFrame original des observations
    - merged_fusion : GeoDataFrame après fusion avec la colonne 'fusion' listant les codes fusionnés
    - cle_geo : nom de la colonne identifiant les cellules (ex: 'codeMaille10Km')
    - cle_ID : nom de la colonne identifiant l'espèce (ex: 'speciesKey')
    - var_obs : nom de la colonne des observations (ex: 'nombreObs')
    
    Retourne :
    - df_biodiv_grouped : DataFrame regroupé avec observations sommées
    """
    # Génération du dictionnaire taxonomique
    df_dico = generer_dictionnaire_taxonomie(df_biodiv, cle_ID)


    # 1. construire le mapping : code original -> code final
    mapping = {}
    for _, row in merged_fusion.iterrows():
        final_cell = row[cle_geo]
        for cell in row['fusion']:
            mapping[cell] = final_cell
    
    mapping_df = pd.DataFrame(list(mapping.items()), columns=['orig', 'final'])
    
    # 2. convertir les types pour compatibilité
    df_biodiv[cle_geo] = df_biodiv[cle_geo].astype(str)
    mapping_df['orig'] = mapping_df['orig'].astype(str)
    
    # 3. merge pour récupérer le code final
    df_merged = df_biodiv.merge(mapping_df, left_on=cle_geo, right_on='orig', how='left')
    
    # 4. remplacer par le code final
    df_merged[cle_geo] = df_merged['final']
    
    # 5. supprimer colonnes intermédiaires
    df_merged = df_merged.drop(columns=['orig', 'final'])
    
    # 6. grouper par cellule, espèce et période, en sommant les observations
    df_grouped = df_merged.groupby([cle_geo, cle_ID, 'periode'], as_index=False)[var_obs].sum()

     # Fusion des données
    df_grouped = pd.merge(df_grouped, df_dico, on=cle_ID)
    
    return df_grouped


def fusionner_depuis_fichiers(
    country_name, grid_size_km, ecosysteme, bornes_temporelles,
    path_data, cle_geo, cle_ID,
    source="GBIF", var_obs="nombreObs",
    seuil_fusion=1000, methode_fusion="barycentre", max_distance=100
):
    """
    Effectue la fusion des mailles à partir des fichiers déjà traités et sauvegardés.

    Parameters
    ----------
    country_name : str
        Nom du pays
    grid_size_km : int
        Taille de la maille en km
    ecosysteme : str
        'terrestre', 'maritime' ou 'combined'
    bornes_temporelles : list
        Bornes de périodes utilisées
    path_data : str
        Répertoire principal de stockage
    cle_geo : str
        Nom de la clé géographique des mailles
    cle_ID : str
        Nom de la clé identifiant unique des enregistrements
    source : str, default "GBIF"
        Source de données
    var_obs : str, default "nombreObs"
        Nom de la variable représentant le nombre d'observations
    seuil_fusion : int, default 1000
        Seuil d'observations pour déclencher la fusion
    methode_fusion : str, default "barycentre"
        Méthode de fusion des cellules
    max_distance : int, default 100
        Distance maximale entre cellules fusionnées

    Returns
    -------
    df_maille_espece : DataFrame
        Données biodiversité après fusion
    grid_fusion : GeoDataFrame
        Grille fusionnée
    """

    # === 1. Charger la grille ===
    grid_path = generate_grid_path(
        path_data, country_name, grid_size_km, ecosysteme, cle_geo, source=source, fusion=False
    )
    grid = gpd.read_file(grid_path)

    # === 2. Charger df_final ===
    chaine_bornes = "_".join(map(str, bornes_temporelles))
    initial_file_path = os.path.join(
        path_data, source,"processed",  f"{source}_{country_name}",
        f"data_{source}_{country_name}_{ecosysteme}_{cle_geo}_{cle_ID}_periodes{chaine_bornes}.csv"
    )
    df_final = pd.read_csv(initial_file_path)

    # === 3. Calcul du nombre d’observations par maille ===
    df_obs = df_final.groupby(cle_geo, as_index=False)[var_obs].sum()
    merged = grid.merge(df_obs, on=cle_geo, how="left")

    merged[var_obs] = merged[var_obs].fillna(0).astype(float)
    merged["coords"] = merged.apply(lambda row: [(row["rank_x"], row["rank_y"])], axis=1)

    print(f"⚡ Fusion des cellules ({methode_fusion}, seuil={seuil_fusion})...")
    merged_fusion = fusion_cells_by_obs(
        merged,
        cle_geo, var_obs,
        seuil=seuil_fusion,
        methode=methode_fusion,
        max_distance=max_distance
    )

    # === 4. Construire la nouvelle grille fusionnée ===
    colonnes_a_garder = [
        "min_lon", "min_lat", "max_lon", "max_lat",
        "rank_x", "rank_y", "cleMaille", cle_geo,
        "country_code", "geometry"
    ]
    grid_fusion = merged_fusion[colonnes_a_garder].copy()

    # Sauvegarde de la grille fusionnée
    grid_fusion_path = generate_grid_path(
        path_data, country_name, grid_size_km, ecosysteme, cle_geo, source=source, fusion=True,seuil_fusion=seuil_fusion
    )
    grid_fusion.to_file(grid_fusion_path, driver="GeoJSON")
    print(f"✅ Grille fusionnée sauvegardée dans : {grid_fusion_path}")

    # === 5. Appliquer la fusion au df_final ===
    df_maille_espece = apply_fusion_to_biodiv(
        df_final, merged_fusion, cle_geo=cle_geo, cle_ID=cle_ID, var_obs=var_obs
    )

    final_file_path_fusion = os.path.join(
        path_data, source,"processed", f"{source}_{country_name}", 
        f"data_{source}_{country_name}_{ecosysteme}_{cle_geo}Adapt{seuil_fusion}_{cle_ID}_periodes{chaine_bornes}.csv"
    )
    df_maille_espece.to_csv(final_file_path_fusion, index=False)
    print(f"✅ Données fusionnées sauvegardées dans : {final_file_path_fusion}")

    return df_maille_espece, grid_fusion


## INAT

In [8]:
import pandas as pd
import os
from datetime import datetime
import pycountry

file_path = r"C:\Users\Aubin\Documents\MANTIS\Data\iNat\iNat_full.csv"

# Détecter le séparateur
with open(file_path, "r", encoding="utf-8") as f:
    first_line = f.readline()
sep = "\t" if "\t" in first_line else ","
print(f"Séparateur détecté : {sep}")

# Colonnes à garder
colonnes_a_garder = ['class', 'decimalLongitude', 'decimalLatitude', 'eventDate', 'year', 'order', 'family', 'species', 'genus', 'occurrenceStatus', 'taxonRank', 'speciesKey', 'occurrenceID', 'countryCode', 'phylum', 'verbatimScientificName', 'kingdom', 'individualCount']
# Dictionnaire ISO2 -> nom pays

today = datetime.today().strftime("%Y%m%d")
base_path = r"C:\Users\Aubin\Documents\MANTIS\DATA\iNat"
chunksize = 1_000_000

# dictionnaire alpha2 -> alpha3
code2to3 = {c.alpha_2: c.alpha_3 for c in pycountry.countries}

# lecture d’un chunk
for chunk_num, chunk in enumerate(pd.read_csv(file_path, sep=sep, chunksize=chunksize, on_bad_lines="skip", encoding="utf-8"), start=1):
    print(f"chunk n°{chunk_num}")
    
    df_reduit = chunk[colonnes_a_garder].copy()
    
    # convertir ISO2 -> ISO3
    df_reduit['countryCode3'] = df_reduit['countryCode'].map(code2to3)
    
    # merge avec shapefile
    df_join = df_reduit.merge(
        world_terrestre[['shapeGroup', 'shapeName']], 
        left_on='countryCode3', 
        right_on='shapeGroup', 
        how='inner'
    )

    for pays, data_pays in df_join.groupby("shapeName"):
        dossier = os.path.join(base_path, "raw",f"iNat_{pays}" )
        os.makedirs(dossier, exist_ok=True)
        
        
        # ajouter le numéro du chunk dans le nom du fichier
        fichier_csv = os.path.join(dossier, f"extractiNat_{pays}_{today}_chunk{chunk_num}.csv")
        data_pays = data_pays.drop(columns=[col for col in ["nomPays", "shapeName"] if col in data_pays.columns])
        
        # écriture CSV
        data_pays.to_csv(fichier_csv, index=False)
        #print(f"Sauvegardé : {fichier_csv}")


FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\Aubin\\Documents\\MANTIS\\Data\\iNat\\iNat_full.csv'

In [ ]:
import pandas as pd
import os
import glob

# Saisie manuelle de la date au format YYYYMMDD
today = "20250914"

base_path = r"C:\Users\Aubin\Documents\MANTIS\DATA\iNat"

# Boucler sur tous les pays connus
for pays in world_terrestre['shapeName']:
    
    dossier = os.path.join(base_path, f"iNat_{pays}", "raw")
    
    # Chercher tous les fichiers chunk de ce pays
    fichiers_temp = glob.glob(os.path.join(dossier, f"extractiNat_{pays}_{today}_chunk*.csv"))
    print(len(fichiers_temp))
    if not fichiers_temp:
        continue
    
    # Lire tous les fichiers temporaires et concaténer
    df_list = [pd.read_csv(f) for f in fichiers_temp]
    df_final = pd.concat(df_list, ignore_index=True)
    
    # Sauvegarder le fichier final
    fichier_final_csv = os.path.join(dossier, f"extractiNat_{pays}_{today}.csv")
    df_final.to_csv(fichier_final_csv, index=False)
    print(f"Fichier final CSV créé : {fichier_final_csv}")
    
    # Supprimer les fichiers temporaires
    for f in fichiers_temp:
        os.remove(f)


In [ ]:
import os
import glob

racine = r"C:\Users\Aubin\Documents\MANTIS\DATA\iNat"

# motif qui parcourt tous les pays -> raw -> sous-dossiers éventuels -> fichiers ciblés
pattern = os.path.join(racine, "*", "raw", "**", "extractiNat_*_20250913.csv")

# recherche récursive
for fichier in glob.glob(pattern, recursive=True):
    try:
        os.remove(fichier)
        print(f"Supprimé : {fichier}")
    except Exception as e:
        print(f"Erreur sur {fichier} : {e}")


## SCRIPT ENTIER

In [10]:

filtered_countries=["Gabon"]

tailles_maille = [10]
cle_ID="speciesKey"
source="GBIF"
n_donnees_par_maille=500
fusion=True

# Bornes temporelles modifiables
bornes_temporelles = [1800, 1990,2010, 2024]  # Vous pouvez les ajuster ici
chaine_bornes = "_".join(map(str, bornes_temporelles))

for country_name in filtered_countries:
    country_name_underscore=country_name.replace('_', ' ')
  
    sig_path = os.path.join(path,'SIG_global')
    path_data = os.path.join(path,'DATA')
    path_fichier = os.path.join(path_data,source,"raw",f"{source}_{country_name}", f"extract{source}_{country_name}.csv")  
    world_terrestre=gpd.read_file(os.path.join(sig_path, "geoBoundariesCGAZ_ADM0.shp"))

    #largeur_recommandee=calcul_largeur_maille_pays(country_name, world_terrestre, path_fichier, 
    
    
    for grid_size_km in tailles_maille:
        cle_geo = f"codeMaille{grid_size_km}Km"
        print(f"\n📢 Traitement pour le pays : {country_name} avec une taille de maille : {grid_size_km} km\n")
        traiter_pays_et_maille(country_name, grid_size_km, bornes_temporelles, path, cle_geo, cle_ID,source=source,
        fusion=fusion,var_obs='nombreObs', seuil_fusion=n_donnees_par_maille, methode_fusion="barycentre", max_distance=100)
    #     rename_sig_files(country_name, largeur_recommandee,path_data,source=source)




📢 Traitement pour le pays : Gabon avec une taille de maille : 10 km

✅ Aucun doublon trouvé dans country_grid_terrestre.
✅ Aucun doublon trouvé dans country_grid_maritime.
✅ Aucun doublon trouvé dans country_grid_combined.
✅ Grid Terrestre trouvée et chargée avec succès !
✅ Grid Maritime trouvée et chargée avec succès !
✅ Grid Combined trouvée et chargée avec succès !
✅ Séparateur choisi : '	' avec 50 colonnes


C:\Users\Aubin\AppData\Local\Temp\ipykernel_16548\3717949375.py:241: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  for df_biodiv in pd.read_csv(



🔍 Traitement du chunk 1...
➡️  En entrée : 15433 espèces, 873288 observations
✅ En sortie : 11533 espèces (-25%)
✅ En sortie : 555158 observations (-36%)
➡️ Association des mailles aux données
363 with several matching grids
32048 with no matching grid
➡️ Association des mailles aux données
24 with several matching grids
457270 with no matching grid
➡️ Association des mailles aux données
387 with several matching grids
817 with no matching grid
⚡ Application de la fusion des cellules (barycentre)...
Fusion n°1, n_min= 0.0
Fusion n°2, n_min= 0.0
Fusion n°3, n_min= 0.0
Fusion n°4, n_min= 0.0
Fusion n°5, n_min= 0.0
Fusion n°6, n_min= 0.0
Fusion n°7, n_min= 0.0
Fusion n°8, n_min= 0.0
Fusion n°9, n_min= 0.0
Fusion n°10, n_min= 0.0
Fusion n°11, n_min= 0.0
Fusion n°12, n_min= 0.0
Fusion n°13, n_min= 0.0
Fusion n°14, n_min= 0.0
Fusion n°15, n_min= 0.0
Fusion n°16, n_min= 0.0
Fusion n°17, n_min= 0.0


C:\Users\Aubin\AppData\Local\Temp\ipykernel_16548\2792791793.py:14: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf['centroid'] = gdf['geometry'].centroid


Fusion n°18, n_min= 0.0
Fusion n°19, n_min= 0.0
Fusion n°20, n_min= 0.0
Fusion n°21, n_min= 0.0
Fusion n°22, n_min= 0.0
Fusion n°23, n_min= 0.0
Fusion n°24, n_min= 0.0
Fusion n°25, n_min= 0.0
Fusion n°26, n_min= 0.0
Fusion n°27, n_min= 0.0
Fusion n°28, n_min= 0.0
Fusion n°29, n_min= 0.0
Fusion n°30, n_min= 0.0
Fusion n°31, n_min= 0.0
Fusion n°32, n_min= 0.0
Fusion n°33, n_min= 0.0
Fusion n°34, n_min= 0.0
Fusion n°35, n_min= 0.0
Fusion n°36, n_min= 0.0
Fusion n°37, n_min= 0.0
Fusion n°38, n_min= 0.0
Fusion n°39, n_min= 0.0
Fusion n°40, n_min= 0.0
Fusion n°41, n_min= 0.0
Fusion n°42, n_min= 0.0
Fusion n°43, n_min= 0.0
Fusion n°44, n_min= 0.0
Fusion n°45, n_min= 0.0
Fusion n°46, n_min= 0.0
Fusion n°47, n_min= 0.0
Fusion n°48, n_min= 0.0
Fusion n°49, n_min= 0.0
Fusion n°50, n_min= 0.0
Fusion n°51, n_min= 0.0
Fusion n°52, n_min= 0.0
Fusion n°53, n_min= 0.0
Fusion n°54, n_min= 0.0
Fusion n°55, n_min= 0.0
Fusion n°56, n_min= 0.0
Fusion n°57, n_min= 0.0
Fusion n°58, n_min= 0.0
Fusion n°59, n_m


### Fusion  depuis fichiers

In [17]:
country_name="France"
grid_size_km = 2
ecosysteme='terrestre'
bornes_temporelles = [1800, 1990,2010, 2024] 
cle_ID="speciesKey"
source="GBIF"
n_donnees_par_maille=5_000
fusion=True
cle_geo = f"codeMaille{grid_size_km}Km"
path_data = os.path.join(path,'DATA')


fusionner_depuis_fichiers(
    country_name, grid_size_km, ecosysteme, bornes_temporelles,
    path_data, cle_geo, cle_ID,
    source="GBIF", var_obs="nombreObs",
    seuil_fusion=n_donnees_par_maille, methode_fusion="barycentre", max_distance=100)

⚡ Fusion des cellules (barycentre, seuil=5000)...
Fusion n°1, n_min= 1000.0
Fusion n°2, n_min= 1000.0


C:\Users\Aubin\AppData\Local\Temp\ipykernel_15132\2792791793.py:14: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf['centroid'] = gdf['geometry'].centroid


Fusion n°3, n_min= 1000.0
Fusion n°4, n_min= 1000.0
Fusion n°5, n_min= 1000.0
Fusion n°6, n_min= 1000.0
Fusion n°7, n_min= 1000.0
Fusion n°8, n_min= 1000.0
Fusion n°9, n_min= 1000.0
Fusion n°10, n_min= 1000.0
Fusion n°11, n_min= 1000.0
Fusion n°12, n_min= 1000.0
Fusion n°13, n_min= 1000.0
Fusion n°14, n_min= 1000.0
Fusion n°15, n_min= 1000.0
Fusion n°16, n_min= 1000.0
Fusion n°17, n_min= 1000.0
Fusion n°18, n_min= 1000.0
Fusion n°19, n_min= 1000.0
Fusion n°20, n_min= 1000.0
Fusion n°21, n_min= 1000.0
Fusion n°22, n_min= 1000.0
Fusion n°23, n_min= 1000.0
Fusion n°24, n_min= 1000.0
Fusion n°25, n_min= 1000.0
Fusion n°26, n_min= 1000.0
Fusion n°27, n_min= 1001.0
Fusion n°28, n_min= 1001.0
Fusion n°29, n_min= 1001.0
Fusion n°30, n_min= 1001.0
Fusion n°31, n_min= 1001.0
Fusion n°32, n_min= 1001.0
Fusion n°33, n_min= 1001.0
Fusion n°34, n_min= 1001.0
Fusion n°35, n_min= 1001.0
Fusion n°36, n_min= 1001.0
Fusion n°37, n_min= 1001.0
Fusion n°38, n_min= 1001.0
Fusion n°39, n_min= 1001.0
Fusion n

(          codeMaille2Km  speciesKey                 periode  nombreObs  \
 0         2km_0_389_FRA     1005007  Période 3: 2011 à 2024          1   
 1         2km_0_389_FRA     1012292  Période 2: 1991 à 2010          1   
 2         2km_0_389_FRA     1035578  Période 2: 1991 à 2010          1   
 3         2km_0_389_FRA     1043097  Période 3: 2011 à 2024          1   
 4         2km_0_389_FRA     1047536  Période 3: 2011 à 2024          1   
 ...                 ...         ...                     ...        ...   
 24161778  2km_9_405_FRA    10694599  Période 2: 1991 à 2010          1   
 24161779  2km_9_405_FRA    10731495  Période 3: 2011 à 2024          1   
 24161780  2km_9_405_FRA    11040384  Période 1: 1801 à 1990          1   
 24161781  2km_9_405_FRA    11071158  Période 3: 2011 à 2024          2   
 24161782  2km_9_405_FRA    12243171  Période 1: 1801 à 1990          1   
 
                             species      vernacularName_fr  \
 0            Rhynchozoon bispinosu